# Tiny Aya → Wazuh alert analyst (AlignTune LoRA)
Runtime: GPU (T4 is enough with 4-bit). Before running: accept the licence on https://huggingface.co/CohereLabs/tiny-aya-global and have a Hugging Face **write** token ready.

Set the two variables in the next cell.

In [ ]:
HF_USER = "Yusheng88"
GITHUB_REPO = "https://github.com/Patupi1/wazuh-alert-analyst"
DATASET = f"{HF_USER}/wazuh-alert-analyst"
ADAPTER_REPO = f"{HF_USER}/tiny-aya-wazuh-analyst-lora"
GGUF_REPO = f"{HF_USER}/tiny-aya-wazuh-analyst-GGUF"

In [ ]:
%pip -q install aligntune auditkit
from huggingface_hub import login; login()
!git clone $GITHUB_REPO repo 2>/dev/null || (cd repo && git pull)
!ls repo

In [ ]:
# QUICK CHECK first: MAX_STEPS = 20 (a few minutes). Then set MAX_STEPS = -1 for the full run (~2-3 h on a T4).
MAX_STEPS = 20
!cd repo && python train/train_sft.py --dataset $DATASET --out /content/out --epochs 2 --max-steps $MAX_STEPS

In [ ]:
# Merge the adapter, quantise to GGUF (for Ollama on the homelab box), push both
!aligntune export merge_adapter /content/out --output /content/out/merged
!aligntune export gguf /content/out/merged --output /content/out/gguf -q Q4_K_M -c llama-cpp
!aligntune export hf_hub /content/out --repo $ADAPTER_REPO --adapter-only
from huggingface_hub import HfApi; api = HfApi(); api.create_repo(GGUF_REPO, exist_ok=True)
api.upload_folder(folder_path='/content/out/gguf', repo_id=GGUF_REPO)
api.upload_file(path_or_fileobj='/content/out/train_summary.json', path_in_repo='train_summary.json', repo_id=ADAPTER_REPO)

Then on the homelab box: `ollama pull hf.co/<HF_USER>/tiny-aya-wazuh-analyst-GGUF:Q4_K_M` and run `src/eval.py` with `--model ollama:hf.co/<HF_USER>/tiny-aya-wazuh-analyst-GGUF:Q4_K_M` on the same test and attack sets as the baseline.